# 🏦 Bangladeshi Banknote Recognition — Crash-Safe Full Pipeline

## Bug Fix Summary (all errors from the original notebook)

| # | Location | Bug | Fix |
|---|---|---|---|
| 1 | Cell 3 `CONFIG` | `train_dir` / `test_dir` keys are **commented out** — `KeyError` when loop injects them | Removed all commented-out path lines; loop injects paths dynamically |
| 2 | Cell 4–6 `PHASE 2` checkpoint block | When a fold is loaded from cache, `fold_l2` and `fold_pca` are **re-computed fresh** on the current split data and used to update `best_fold_l2/pca` — but these objects were never **saved to the checkpoint**. After a restart the loaded best fold's preprocessors are wrong | Fixed: `fold_l2` and `fold_pca` are now **serialised inside the fold checkpoint pkl** and restored correctly on reload |
| 3 | Cell 4–6 `PHASE 3` final model | `final_model.fit()` is **always called** even when `FINAL_MODEL_PATH` already exists — retrains every run | Fixed: added `if os.path.exists(FINAL_MODEL_PATH)` guard to skip training and load the saved model |
| 4 | Cell 4–6 `PHASE 3` final model | `train_preds_final`, `val_preds_final`, `train_preds_proba_final`, `val_preds_proba_final`, `test_image_info_final`, `y_true_test_encoded`, `y_pred_test_encoded` are **never assigned** — `NameError` in Cells 7-11 | Fixed: all variables assigned correctly from model predictions |
| 5 | Cell 6 summary | `ALL_RESULTS.items()++` — **SyntaxError** (`++` is not Python) | Removed `++` |
| 6 | Cell 8 SHAP | `SHAP_FILE` path is **hardcoded** to a single global path — datasets overwrite each other's SHAP values | Fixed: path is `DS_CKPT_DIR/shap_values.npy` per dataset |
| 7 | Cell 10 PCA report | `all_fold_data` referenced **after the loop** without being in scope for Cell 10 if run standalone | Fixed: persisted in `ALL_RESULTS[ds_name]` and re-read from there |
| 8 | All cells | No **XLSX summary** of K-Fold + final model metrics across all datasets | Added: Cell 12 writes a formatted multi-sheet `.xlsx` to Drive |
| 9 | Cell 4–6 | K-Fold `kfold_results.csv` and final `final_model_metrics.csv` saved per-dataset but **no combined xlsx** | Fixed: combined xlsx written at end with one sheet per dataset + summary sheet |

---

## How crash-recovery works

Every expensive result is saved to **Google Drive** immediately after computation.  
On restart, run all cells top-to-bottom — completed steps are skipped instantly.

```
Drive layout
└── banknote_checkpoints/
    ├── Primary_Dataset-1/
    │   ├── X_raw.npy              ← CNN features (most expensive)
    │   ├── y_raw.npy
    │   ├── train_paths.npy
    │   ├── folds/
    │   │   ├── fold1_metrics.pkl  ← fold results + fold_l2 + fold_pca
    │   │   └── ...fold5
    │   ├── final_model.h5
    │   ├── final_l2.pkl
    │   ├── final_pca.pkl
    │   ├── final_results.pkl      ← all metrics + predictions
    │   ├── shap_values.npy
    │   └── kfold_results.csv
    ├── Primary_Dataset-2/ ...
    └── ALL_DATASETS_SUMMARY.xlsx  ← combined metrics workbook
```


---
## Cell 0 — Mount Google Drive
Must run first. Drive is the only storage that survives Colab disconnections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted.")


---
## Cell 1 — Extract Datasets (skips if already extracted)

In [ ]:
import zipfile, os

DATASETS = [
    {
        "name"      : "Primary_Dataset-1",
        "zip_path"  : "/content/drive/MyDrive/simpleBGNEW.zip",
        "train_dir" : "/content/bd_money/simple_Bangladeshi_dataset_split_final/train",
        "test_dir"  : "/content/bd_money/simple_Bangladeshi_dataset_split_final/test",
    },
    {
        "name"      : "Primary_Dataset-2",
        "zip_path"  : "/content/drive/MyDrive/complexBGNEW.zip",
        "train_dir" : "/content/bd_money/complex_bangladeshi_dataset_split_final/train",
        "test_dir"  : "/content/bd_money/complex_bangladeshi_dataset_split_final/test",
    },
    {
        "name"      : "Custom_Dataset-1",
        "zip_path"  : "/content/drive/MyDrive/kaggleNEW.zip",
        "train_dir" : "/content/bd_money/kaggle_split_final/train",
        "test_dir"  : "/content/bd_money/kaggle_split_final/test",
    },
    {
        "name"      : "Custom_Dataset-2",
        "zip_path"  : "/content/drive/MyDrive/kaggle_originalNEW.zip",
        "train_dir" : "/content/bd_money/kaggle_original_split_final/train",
        "test_dir"  : "/content/bd_money/kaggle_original_split_final/test",
    },
    {
        "name"      : "Custom_Dataset-3",
        "zip_path"  : "/content/drive/MyDrive/finalNew.zip",
        "train_dir" : "/content/bd_money/merged_data_split_final/train",
        "test_dir"  : "/content/bd_money/merged_data_split_final/test",
    },
]

extract_path = "/content/bd_money"
os.makedirs(extract_path, exist_ok=True)

for ds in DATASETS:
    if os.path.isdir(ds["train_dir"]):
        print(f"  ⏭️  {ds['name']} already extracted — skipping.")
        continue
    print(f"📦 Extracting {ds['name']} ...")
    with zipfile.ZipFile(ds["zip_path"], "r") as zf:
        zf.extractall(extract_path)
    print(f"  ✅ Done.")

print("\n✅ All datasets ready at:", extract_path)


---
## Cell 2 — Install Dependencies & Imports

In [ ]:
!pip install lime shap openpyxl --quiet

import os, cv2, pickle, time, platform, psutil, tracemalloc, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from itertools import cycle
from collections import defaultdict
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, Normalizer, label_binarize
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef,
    roc_auc_score, roc_curve, auc, classification_report,
    precision_recall_fscore_support
)

import lime
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from skimage.segmentation import mark_boundaries
import shap

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Large, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras import regularizers

np.random.seed(42); random.seed(42); tf.random.set_seed(42)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")


---
## Cell 3 — Configuration, Pre-trained Backbones & All Helper Functions

> **Bug fix #1 applied here:** The original `CONFIG` dict had all `train_dir`/`test_dir`
> lines commented out (one had a double `#`). This caused a `KeyError` when the main loop
> tried to inject dataset paths. Fixed by removing all commented-out path entries — the
> loop sets them dynamically.


In [ ]:
# ── BUG FIX #1: Removed all commented-out train_dir/test_dir lines.
# The dataset loop (Cell 4) injects CONFIG["train_dir"] and CONFIG["test_dir"]
# at the start of each iteration, so no path should live here.
CONFIG = {
    # train_dir and test_dir are set by the dataset loop — do NOT add them here
    'n_folds'               : 5,
    'lime_samples'          : 1000,
    'lime_examples_per_fold': 1,
    'input_shape'           : (224, 224, 3),
    'random_state'          : 42,
}

# ── Augmentors ────────────────────────────────────────────────────────────────
augmentor = ImageDataGenerator(
    rotation_range=30, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.1, zoom_range=[0.8,1.2], brightness_range=[0.8,1.2],
    horizontal_flip=True, fill_mode='nearest',
)
tta_augmentor = ImageDataGenerator(
    rotation_range=15, width_shift_range=0.05, height_shift_range=0.05,
    zoom_range=[0.9,1.1], horizontal_flip=True, fill_mode='nearest',
)

# ── CNN backbones ─────────────────────────────────────────────────────────────
print("Loading pre-trained CNN backbones...")
mobilenet_model = MobileNetV3Large(
    weights='imagenet', include_top=False, pooling='avg',
    input_shape=CONFIG['input_shape'])
efficientnet_model = EfficientNetB0(
    weights='imagenet', include_top=False, pooling='avg',
    input_shape=CONFIG['input_shape'])
for layer in mobilenet_model.layers[:-20]:   layer.trainable = False
for layer in efficientnet_model.layers[:-20]: layer.trainable = False
print("✅ CNN backbones ready.")


# ─────────────────────────────────────────────────────────────────────────────
def extract_single_image_features(image):
    mob = mobilenet_model.predict(
        np.expand_dims(mobilenet_preprocess(image.copy()), axis=0), verbose=0)
    eff = efficientnet_model.predict(
        np.expand_dims(efficientnet_preprocess(image.copy()), axis=0), verbose=0)
    return np.concatenate((mob.flatten(), eff.flatten()))


def extract_features_from_directory(directory_path, augment_data=True):
    features, labels, image_paths = [], [], []
    if not os.path.exists(directory_path):
        raise FileNotFoundError(f"Directory not found: {directory_path}")
    class_folders = sorted([
        f for f in os.listdir(directory_path)
        if os.path.isdir(os.path.join(directory_path, f))])
    for label in tqdm(class_folders, desc="Extracting features"):
        class_dir = os.path.join(directory_path, label)
        for img_name in sorted(os.listdir(class_dir)):
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            try:
                orig = cv2.resize(img, CONFIG['input_shape'][:2])
                features.append(extract_single_image_features(orig))
                labels.append(label); image_paths.append(img_path)
                if augment_data:
                    aug = augmentor.random_transform(orig)
                    features.append(extract_single_image_features(aug))
                    labels.append(label); image_paths.append(img_path + "_aug")
            except Exception as e:
                print(f"Error: {img_path}: {e}")
    return np.array(features), np.array(labels), image_paths


def verify_no_train_test_overlap(train_dir, test_dir):
    train_files, test_files = set(), set()
    for cls in os.listdir(train_dir):
        p = os.path.join(train_dir, cls)
        if os.path.isdir(p):
            for f in os.listdir(p): train_files.add(f)
    if os.path.isdir(test_dir):
        for f in os.listdir(test_dir): test_files.add(f)
    overlap = train_files & test_files
    if overlap: print(f"⚠️  WARNING: {len(overlap)} filenames in BOTH splits!")
    else:        print(f"✅ No train/test overlap. Train={len(train_files)}, Test={len(test_files)}")
    return len(overlap) == 0


def create_model(input_dim, num_classes, label_smoothing=0.0,
                 learning_rate=1e-4, weight_decay=1e-4):
    l2 = regularizers.l2(1e-4)
    model = Sequential([
        Dense(512, activation='relu', kernel_regularizer=l2, input_shape=(input_dim,)),
        BatchNormalization(), Dropout(0.5),
        Dense(256, activation='relu', kernel_regularizer=l2),
        BatchNormalization(), Dropout(0.4),
        Dense(128, activation='relu', kernel_regularizer=l2),
        BatchNormalization(), Dropout(0.3),
        Dense(64,  activation='relu', kernel_regularizer=l2),
        BatchNormalization(), Dropout(0.2),
        Dense(num_classes, activation='softmax'),
    ])
    if label_smoothing > 0:
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, weight_decay=weight_decay),
            loss=CategoricalCrossentropy(label_smoothing=label_smoothing),
            metrics=['accuracy'])
    else:
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, weight_decay=weight_decay),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


def calculate_comprehensive_metrics(y_true, y_pred, y_pred_proba=None):
    metrics = {
        "Accuracy"    : accuracy_score(y_true, y_pred),
        "Precision"   : precision_score(y_true, y_pred, average='weighted', zero_division=0),
        "Recall"      : recall_score(y_true, y_pred,    average='weighted', zero_division=0),
        "F1"          : f1_score(y_true, y_pred,        average='weighted', zero_division=0),
        "Cohen_Kappa" : cohen_kappa_score(y_true, y_pred),
        "MCC"         : matthews_corrcoef(y_true, y_pred),
    }
    if y_pred_proba is not None:
        try:
            metrics["AUC"] = (roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='weighted')
                              if len(np.unique(y_true)) > 2
                              else roc_auc_score(y_true, y_pred_proba[:, 1]))
        except Exception:
            metrics["AUC"] = 0.0
    else:
        metrics["AUC"] = 0.0
    return metrics


def evaluate_on_test_set(model, test_dir, fold_l2, fold_pca, label_encoder):
    y_true, y_pred, y_proba, info = [], [], [], []
    test_files = sorted([f for f in os.listdir(test_dir) if "_" in f])
    for fn in tqdm(test_files, desc="Testing", leave=False):
        true_label = fn.split("_")[0]
        img = cv2.imread(os.path.join(test_dir, fn))
        if img is None: continue
        try:
            img_r   = cv2.resize(img, CONFIG['input_shape'][:2])
            feat    = fold_pca.transform(fold_l2.transform([extract_single_image_features(img_r)]))
            probs   = model.predict(feat, verbose=0)[0]
            pred    = label_encoder.inverse_transform([np.argmax(probs)])[0]
            y_true.append(true_label); y_pred.append(pred); y_proba.append(probs)
            info.append({'path': os.path.join(test_dir, fn), 'name': fn,
                         'true_label': true_label, 'pred_label': pred,
                         'confidence': float(np.max(probs))})
        except Exception as e:
            print(f"  Error {fn}: {e}")
    return y_true, y_pred, np.array(y_proba), info


def evaluate_on_test_set_tta(model, test_dir, fold_l2, fold_pca, label_encoder, n_augments=5):
    y_true, y_pred, y_proba, info = [], [], [], []
    test_files = sorted([f for f in os.listdir(test_dir) if "_" in f])
    for fn in tqdm(test_files, desc=f"TTA Testing (n={n_augments})", leave=False):
        true_label = fn.split("_")[0]
        img = cv2.imread(os.path.join(test_dir, fn))
        if img is None: continue
        try:
            img_r  = cv2.resize(img, CONFIG['input_shape'][:2])
            pall   = [model.predict(fold_pca.transform(fold_l2.transform(
                          [extract_single_image_features(img_r)])), verbose=0)[0]]
            for _ in range(n_augments):
                a = tta_augmentor.random_transform(img_r)
                pall.append(model.predict(fold_pca.transform(fold_l2.transform(
                    [extract_single_image_features(a)])), verbose=0)[0])
            avg   = np.mean(pall, axis=0)
            pred  = label_encoder.inverse_transform([np.argmax(avg)])[0]
            y_true.append(true_label); y_pred.append(pred); y_proba.append(avg)
            info.append({'path': os.path.join(test_dir, fn), 'name': fn,
                         'true_label': true_label, 'pred_label': pred,
                         'confidence': float(np.max(avg))})
        except Exception as e:
            print(f"  TTA error {fn}: {e}")
    return y_true, y_pred, np.array(y_proba), info


def generate_lime_explanation(model, img_path, fold_l2, fold_pca, label_encoder, num_samples=1000):
    try:
        img = cv2.imread(img_path)
        if img is None: return None, None, None
        img   = cv2.resize(img, CONFIG['input_shape'][:2])
        img_r = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        def pred_fn(imgs):
            preds = []
            for im in imgs:
                try:
                    bgr  = cv2.cvtColor(im, cv2.COLOR_RGB2BGR)
                    r    = cv2.resize(bgr, CONFIG['input_shape'][:2])
                    feat = fold_pca.transform(fold_l2.transform([extract_single_image_features(r)]))
                    preds.append(model.predict(feat, verbose=0)[0])
                except Exception:
                    preds.append(np.ones(len(label_encoder.classes_)) / len(label_encoder.classes_))
            return np.array(preds)
        explainer   = lime_image.LimeImageExplainer()
        explanation = explainer.explain_instance(
            img_r, pred_fn,
            top_labels=min(5, len(label_encoder.classes_)),
            hide_color=0, num_samples=num_samples,
            segmentation_fn=SegmentationAlgorithm(
                'quickshift', kernel_size=4, max_dist=200, ratio=0.2))
        return explanation, img_r, img
    except Exception as e:
        print(f"LIME error: {e}"); return None, None, None


def create_all_folds_radar_plot(all_fold_data, ds_name):
    radar_metrics = ['Accuracy','Precision','Recall','F1','Cohen_Kappa','MCC','AUC']
    max_val, max_test, bv, bt = {},{},{},{}
    for m in radar_metrics:
        mv, mt, bvv, btt = -1e9, -1e9, 1, 1
        for fd in all_fold_data:
            if fd['val_metrics'][m]  > mv: mv,  bvv = fd['val_metrics'][m],  fd['fold']
            if fd['test_metrics'][m] > mt: mt,  btt = fd['test_metrics'][m], fd['fold']
        max_val[m]=mv; max_test[m]=mt; bv[m]=bvv; bt[m]=btt
    angles = np.linspace(0, 2*np.pi, len(radar_metrics), endpoint=False).tolist() + [0]
    vc = plt.cm.Blues(np.linspace(0.4,0.9,CONFIG['n_folds']))
    tc = plt.cm.Reds(np.linspace(0.4,0.9,CONFIG['n_folds']))
    fig, (ax1,ax2) = plt.subplots(1,2,figsize=(24,12),subplot_kw=dict(projection='polar'))
    for ax, colors, key, tpfx, lc, mx, bst in [
        (ax1,vc,'val_metrics','Validation','navy',max_val,bv),
        (ax2,tc,'test_metrics','Test','darkred',max_test,bt)]:
        ax.set_title(f'{tpfx} Metrics — All Folds', fontsize=13, fontweight='bold', pad=30)
        for fd in all_fold_data:
            vals = [((v+1)/2 if m in ['Cohen_Kappa','MCC'] else v)
                    for m,v in [(m,fd[key][m]) for m in radar_metrics]]
            vals += vals[:1]
            ax.plot(angles, vals, 'o-', lw=2, ms=6,
                    label=f'Fold {fd["fold"]}', color=colors[fd['fold']-1], alpha=0.7)
            ax.fill(angles, vals, alpha=0.1, color=colors[fd['fold']-1])
        for i, angle in enumerate(angles[:-1]):
            orig = mx[radar_metrics[i]]
            norm = (orig+1)/2 if radar_metrics[i] in ['Cohen_Kappa','MCC'] else orig
            ax.text(angle, norm+0.12, f'{orig:.3f}\n(F{bst[radar_metrics[i]]})',
                    ha='center', va='center', fontsize=9, fontweight='bold', color=lc,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='white',
                              alpha=0.9, edgecolor=lc, lw=2))
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(radar_metrics, fontsize=12, fontweight='bold')
        ax.set_ylim(0,1.3); ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
        ax.grid(True, alpha=0.3, lw=1.5)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3,1.1), fontsize=10)
    plt.suptitle(f'Radar — All {CONFIG["n_folds"]} Folds | {ds_name}', fontsize=15, fontweight='bold')
    plt.tight_layout(); plt.show()

print("✅ All helper functions defined.")


---
## Cell 4 — Main Pipeline: Feature Extraction + K-Fold CV + Final Model (all 5 datasets)

> **Bug fixes applied in this cell:**
> - **#2** Fold checkpoint now serialises `fold_l2` and `fold_pca` inside the pkl so they are correctly restored on reload — previously the checkpoint loaded stale preprocessors re-computed on the current split.
> - **#3** Final model training is guarded by `if os.path.exists(FINAL_MODEL_PATH)` — previously it retrained every run.
> - **#4** All downstream variables (`train_preds_final`, `val_preds_final`, `train_preds_proba_final`, `val_preds_proba_final`, `test_image_info_final`, `y_true_test_encoded`, `y_pred_test_encoded`) are now correctly assigned in both the "load from cache" and "train fresh" branches.
> - **#5** `ALL_RESULTS.items()++` → `ALL_RESULTS.items()` (SyntaxError removed).


In [ ]:
import numpy as np, os, pickle, pandas as pd
from sklearn.preprocessing import Normalizer, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, train_test_split
from tensorflow.keras.models import load_model

CKPT_ROOT = "/content/drive/MyDrive/hybrid-adamW-1e-3-0.5-0.4-0.3-0.2"
os.makedirs(CKPT_ROOT, exist_ok=True)

ALL_RESULTS = {}   # accumulates per-dataset summary for the final xlsx

for ds in DATASETS:
    ds_name = ds["name"]
    print("\n" + "█"*80)
    print(f"  DATASET: {ds_name.upper()}")
    print("█"*80)

    # ── BUG FIX #1: inject paths dynamically ─────────────────────────────────
    CONFIG["train_dir"] = ds["train_dir"]
    CONFIG["test_dir"]  = ds["test_dir"]

    DS_CKPT_DIR   = os.path.join(CKPT_ROOT, ds_name)
    FEATURES_FILE = os.path.join(DS_CKPT_DIR, "X_raw.npy")
    LABELS_FILE   = os.path.join(DS_CKPT_DIR, "y_raw.npy")
    PATHS_FILE    = os.path.join(DS_CKPT_DIR, "train_paths.npy")
    FOLD_CKPT_DIR = os.path.join(DS_CKPT_DIR, "folds")
    FINAL_MODEL_PATH  = os.path.join(DS_CKPT_DIR, "final_model.h5")
    FINAL_L2_PATH     = os.path.join(DS_CKPT_DIR, "final_l2.pkl")
    FINAL_PCA_PATH    = os.path.join(DS_CKPT_DIR, "final_pca.pkl")
    FINAL_RESULTS_PKL = os.path.join(DS_CKPT_DIR, "final_results.pkl")
    os.makedirs(DS_CKPT_DIR,   exist_ok=True)
    os.makedirs(FOLD_CKPT_DIR, exist_ok=True)

    if not os.path.isdir(CONFIG["train_dir"]):
        print(f"  ⚠️  Train dir not found — skipping: {CONFIG['train_dir']}")
        continue

    # ══════════════════════════════════════════════════════════
    # PHASE 1 — CNN Feature Extraction
    # ══════════════════════════════════════════════════════════
    verify_no_train_test_overlap(CONFIG["train_dir"], CONFIG["test_dir"])

    if os.path.exists(FEATURES_FILE) and os.path.exists(LABELS_FILE):
        print(f"[{ds_name}] 📂 Loading features from checkpoint...")
        X_raw       = np.load(FEATURES_FILE, allow_pickle=True)
        y_raw       = np.load(LABELS_FILE,   allow_pickle=True)
        train_paths = list(np.load(PATHS_FILE, allow_pickle=True))
        print(f"[{ds_name}] ✅ X_raw={X_raw.shape}")
    else:
        print(f"[{ds_name}] 🔄 Extracting features (may take 20-40 min)...")
        X_raw, y_raw, train_paths = extract_features_from_directory(
            CONFIG["train_dir"], augment_data=True)
        np.save(FEATURES_FILE, X_raw)
        np.save(LABELS_FILE,   y_raw)
        np.save(PATHS_FILE,    np.array(train_paths, dtype=object))
        print(f"[{ds_name}] ✅ Features saved. X_raw={X_raw.shape}")

    label_encoder = LabelEncoder()
    y_encoded     = label_encoder.fit_transform(y_raw)
    print(f"[{ds_name}] Classes ({len(label_encoder.classes_)}): {label_encoder.classes_}")

    # ══════════════════════════════════════════════════════════
    # PHASE 2 — K-Fold Cross-Validation
    # ══════════════════════════════════════════════════════════
    kf                = KFold(n_splits=CONFIG["n_folds"], shuffle=True,
                               random_state=CONFIG["random_state"])
    results           = []
    all_fold_data     = []
    best_val_accuracy = 0.0
    best_fold_idx     = 1
    best_fold_l2      = None
    best_fold_pca     = None

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_raw, y_encoded), 1):
        fold_metrics_path = os.path.join(FOLD_CKPT_DIR, f"fold{fold}_metrics.pkl")
        fold_model_path   = os.path.join(FOLD_CKPT_DIR, f"fold{fold}_best_model.h5")

        # ── BUG FIX #2: checkpoint now stores fold_l2 + fold_pca ─────────────
        if os.path.exists(fold_metrics_path):
            with open(fold_metrics_path, "rb") as fp:
                saved = pickle.load(fp)
            print(f"[{ds_name}] ⏭️  Fold {fold} loaded from checkpoint. "
                  f"Val Acc={saved['fold_data']['val_metrics']['Accuracy']:.4f}")
            results.append(saved["fold_results"])
            all_fold_data.append(saved["fold_data"])
            # Restore the correct preprocessors from the checkpoint (BUG FIX #2)
            if saved["fold_data"]["val_metrics"]["Accuracy"] > best_val_accuracy:
                best_val_accuracy = saved["fold_data"]["val_metrics"]["Accuracy"]
                best_fold_idx     = fold
                best_fold_l2      = saved["fold_data"]["fold_l2"]   # from pkl
                best_fold_pca     = saved["fold_data"]["fold_pca"]  # from pkl
            continue

        # ── Train fold ────────────────────────────────────────────────────────
        print(f"\n[{ds_name}] {'='*50}")
        print(f"[{ds_name}] FOLD {fold}/{CONFIG['n_folds']}")
        print(f"[{ds_name}] {'='*50}")

        X_tr_raw = X_raw[train_idx];  X_va_raw = X_raw[val_idx]
        y_tr     = y_encoded[train_idx]; y_va   = y_encoded[val_idx]

        fold_l2  = Normalizer(norm="l2")
        X_tr_n   = fold_l2.fit_transform(X_tr_raw)
        X_va_n   = fold_l2.transform(X_va_raw)

        fold_pca = PCA(n_components=0.95, random_state=CONFIG["random_state"])
        X_tr_p   = fold_pca.fit_transform(X_tr_n)
        X_va_p   = fold_pca.transform(X_va_n)
        print(f"  PCA components: {fold_pca.n_components_}")

        model = create_model(X_tr_p.shape[1], len(np.unique(y_encoded)),
                             label_smoothing=0.0, learning_rate=1e-4, weight_decay=1e-4)
        callbacks = [
            EarlyStopping(monitor="val_loss", patience=7,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                              patience=4, min_lr=1e-6, verbose=1),
            ModelCheckpoint(fold_model_path, monitor="val_loss",
                            save_best_only=True, verbose=0),
        ]
        model.fit(X_tr_p, y_tr, validation_data=(X_va_p, y_va),
                  epochs=50, batch_size=32, verbose=1, callbacks=callbacks)

        tr_proba = model.predict(X_tr_p, verbose=0)
        tr_m     = calculate_comprehensive_metrics(y_tr, np.argmax(tr_proba,1), tr_proba)
        va_proba = model.predict(X_va_p, verbose=0)
        va_m     = calculate_comprehensive_metrics(y_va, np.argmax(va_proba,1), va_proba)

        yt_t, yp_t, ypr_t, _ = evaluate_on_test_set(
            model, CONFIG["test_dir"], fold_l2, fold_pca, label_encoder)
        te_m = calculate_comprehensive_metrics(
            label_encoder.transform(yt_t), label_encoder.transform(yp_t), ypr_t)

        print(f"\n  {'Metric':<18} {'Train':>10} {'Val':>10} {'Test':>10}")
        for m in ["Accuracy","Precision","Recall","F1","Cohen_Kappa","MCC","AUC"]:
            print(f"  {m:<18} {tr_m[m]:>10.4f} {va_m[m]:>10.4f} {te_m[m]:>10.4f}")

        fold_results = {"Fold": fold}
        fold_results.update({f"Train_{k}": v for k,v in tr_m.items()})
        fold_results.update({f"Val_{k}":   v for k,v in va_m.items()})
        fold_results.update({f"Test_{k}":  v for k,v in te_m.items()})
        results.append(fold_results)

        # BUG FIX #2: store fold_l2 and fold_pca inside the pkl
        fold_data = {
            "fold": fold,
            "train_metrics": tr_m, "val_metrics": va_m, "test_metrics": te_m,
            "fold_l2": fold_l2,    "fold_pca": fold_pca,   # ← FIX
        }
        all_fold_data.append(fold_data)

        with open(fold_metrics_path, "wb") as fp:
            pickle.dump({"fold_results": fold_results, "fold_data": fold_data}, fp)
        print(f"[{ds_name}] 💾 Fold {fold} checkpoint saved.")

        if va_m["Accuracy"] > best_val_accuracy:
            best_val_accuracy = va_m["Accuracy"]
            best_fold_idx     = fold
            best_fold_l2      = fold_l2
            best_fold_pca     = fold_pca

    # K-Fold summary
    df_results = pd.DataFrame(results)
    avg = df_results.mean(numeric_only=True)
    std = df_results.std(numeric_only=True)
    print(f"\n[{ds_name}] K-FOLD SUMMARY")
    print(f"  {'Metric':<18} {'Train Mean±Std':>18} {'Val Mean±Std':>16} {'Test Mean±Std':>16}")
    for m in ["Accuracy","Precision","Recall","F1","Cohen_Kappa","MCC","AUC"]:
        print(f"  {m:<18} {avg[f'Train_{m}']:>8.4f}±{std[f'Train_{m}']:.3f}  "
              f"{avg[f'Val_{m}']:>8.4f}±{std[f'Val_{m}']:.3f}  "
              f"{avg[f'Test_{m}']:>8.4f}±{std[f'Test_{m}']:.3f}")
    df_results.to_csv(os.path.join(DS_CKPT_DIR, "kfold_results.csv"), index=False)

    # K-Fold per-metric line charts
    metrics_plot = ["Accuracy","Precision","Recall","F1","Cohen_Kappa","MCC","AUC"]
    fig, axes = plt.subplots(3,3,figsize=(24,15)); axes = axes.ravel()
    for idx, m in enumerate(metrics_plot):
        ax = axes[idx]
        tr_v = df_results[f'Train_{m}'].values
        va_v = df_results[f'Val_{m}'].values
        te_v = df_results[f'Test_{m}'].values
        fv   = df_results['Fold'].values
        ax.plot(fv, tr_v, 'go-', lw=2, ms=8, label='Train')
        ax.plot(fv, va_v, 'bo-', lw=2, ms=8, label='Val')
        ax.plot(fv, te_v, 'ro-', lw=2, ms=8, label='Test')
        ax.plot(fv[np.argmax(tr_v)], tr_v.max(), 'g*', ms=16, label=f'Max Tr:{tr_v.max():.3f}')
        ax.plot(fv[np.argmax(va_v)], va_v.max(), 'b*', ms=16, label=f'Max Va:{va_v.max():.3f}')
        ax.plot(fv[np.argmax(te_v)], te_v.max(), 'r*', ms=16, label=f'Max Te:{te_v.max():.3f}')
        ax.set_title(f'{m} per Fold', fontweight='bold'); ax.set_xlabel('Fold'); ax.set_ylabel(m)
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
        ax.set_ylim((-1,1) if m in ['Cohen_Kappa','MCC'] else (0,1))
    for i in range(len(metrics_plot), len(axes)): axes[i].axis('off')
    plt.suptitle(f'K-Fold Metrics — {ds_name}', fontsize=16, fontweight='bold')
    plt.tight_layout(); plt.show()
    create_all_folds_radar_plot(all_fold_data, ds_name)

    # ══════════════════════════════════════════════════════════
    # PHASE 3 — Final Model Training
    # BUG FIX #3: guard with os.path.exists(FINAL_MODEL_PATH)
    # BUG FIX #4: all downstream variables set in both branches
    # ══════════════════════════════════════════════════════════
    print(f"\n[{ds_name}] Final model...")

    X_tr_raw_f, X_va_raw_f, y_train_final, y_val_final = train_test_split(
        X_raw, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

    # Load or fit L2 + PCA
    if os.path.exists(FINAL_L2_PATH) and os.path.exists(FINAL_PCA_PATH):
        with open(FINAL_L2_PATH,"rb") as f:  final_l2  = pickle.load(f)
        with open(FINAL_PCA_PATH,"rb") as f: final_pca = pickle.load(f)
        X_train_final = final_pca.transform(final_l2.transform(X_tr_raw_f))
        X_val_final   = final_pca.transform(final_l2.transform(X_va_raw_f))
    else:
        final_l2  = Normalizer(norm="l2")
        final_pca = PCA(n_components=0.95, random_state=42)
        X_train_final = final_pca.fit_transform(final_l2.fit_transform(X_tr_raw_f))
        X_val_final   = final_pca.transform(final_l2.transform(X_va_raw_f))
        with open(FINAL_L2_PATH,"wb")  as f: pickle.dump(final_l2,  f)
        with open(FINAL_PCA_PATH,"wb") as f: pickle.dump(final_pca, f)
    print(f"  PCA components (final): {final_pca.n_components_}")

    # ── BUG FIX #3: check if final results are already cached ─────────────────
    if os.path.exists(FINAL_RESULTS_PKL) and os.path.exists(FINAL_MODEL_PATH):
        print(f"[{ds_name}] 📂 Loading final model results from checkpoint...")
        final_model = load_model(FINAL_MODEL_PATH)
        with open(FINAL_RESULTS_PKL,"rb") as f:
            fr = pickle.load(f)
        # ── BUG FIX #4: unpack ALL variables needed by later cells ───────────
        train_preds_proba_final  = fr["train_preds_proba_final"]
        train_preds_final        = fr["train_preds_final"]
        train_metrics_final      = fr["train_metrics_final"]
        val_preds_proba_final    = fr["val_preds_proba_final"]
        val_preds_final          = fr["val_preds_final"]
        val_metrics_final        = fr["val_metrics_final"]
        y_pred_proba_test        = fr["y_pred_proba_test"]
        y_pred_test_encoded      = fr["y_pred_test_encoded"]
        y_true_test_encoded      = fr["y_true_test_encoded"]
        test_metrics_final       = fr["test_metrics_final"]
        test_image_info_final    = fr["test_image_info_final"]
        history_dict             = fr["history_dict"]
        print(f"[{ds_name}] ✅ Loaded from cache.")

    else:
        # ── Train fresh ───────────────────────────────────────────────────────
        final_model = create_model(
            X_train_final.shape[1], len(label_encoder.classes_),
            label_smoothing=0.0, learning_rate=1e-4, weight_decay=1e-4)

        history_obj = final_model.fit(
            X_train_final, y_train_final,
            validation_data=(X_val_final, y_val_final),
            epochs=50, batch_size=32, verbose=1,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=7,
                              restore_best_weights=True, verbose=1),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=4, min_lr=1e-6, verbose=1),
                ModelCheckpoint(FINAL_MODEL_PATH, monitor="val_loss",
                                save_best_only=True, verbose=0),
            ])
        history_dict = history_obj.history

        # ── BUG FIX #4: assign all variables ─────────────────────────────────
        train_preds_proba_final = final_model.predict(X_train_final, verbose=0)
        train_preds_final       = np.argmax(train_preds_proba_final, axis=1)
        train_metrics_final     = calculate_comprehensive_metrics(
            y_train_final, train_preds_final, train_preds_proba_final)

        val_preds_proba_final   = final_model.predict(X_val_final, verbose=0)
        val_preds_final         = np.argmax(val_preds_proba_final, axis=1)
        val_metrics_final       = calculate_comprehensive_metrics(
            y_val_final, val_preds_final, val_preds_proba_final)

        y_true_t, y_pred_t, y_proba_t, test_image_info_final = evaluate_on_test_set_tta(
            final_model, CONFIG["test_dir"], final_l2, final_pca, label_encoder, n_augments=5)
        y_true_test_encoded  = label_encoder.transform(y_true_t)
        y_pred_test_encoded  = label_encoder.transform(y_pred_t)
        y_pred_proba_test    = y_proba_t
        test_metrics_final   = calculate_comprehensive_metrics(
            y_true_test_encoded, y_pred_test_encoded, y_pred_proba_test)

        # Save everything needed by downstream cells
        with open(FINAL_RESULTS_PKL,"wb") as f:
            pickle.dump({
                "train_preds_proba_final"  : train_preds_proba_final,
                "train_preds_final"        : train_preds_final,
                "train_metrics_final"      : train_metrics_final,
                "val_preds_proba_final"    : val_preds_proba_final,
                "val_preds_final"          : val_preds_final,
                "val_metrics_final"        : val_metrics_final,
                "y_pred_proba_test"        : y_pred_proba_test,
                "y_pred_test_encoded"      : y_pred_test_encoded,
                "y_true_test_encoded"      : y_true_test_encoded,
                "test_metrics_final"       : test_metrics_final,
                "test_image_info_final"    : test_image_info_final,
                "history_dict"             : history_dict,
            }, f)
        print(f"[{ds_name}] 💾 Final model results saved.")

    # Print final metrics
    print(f"\n[{ds_name}] FINAL MODEL METRICS")
    print(f"  {'Metric':<18} {'Train':>10} {'Val':>10} {'Test (TTA)':>12}")
    for m in ["Accuracy","Precision","Recall","F1","Cohen_Kappa","MCC","AUC"]:
        print(f"  {m:<18} {train_metrics_final[m]:>10.4f} "
              f"{val_metrics_final[m]:>10.4f} {test_metrics_final[m]:>12.4f}")

    pd.DataFrame([{
        "Dataset": ds_name,
        **{f"Train_{k}": v for k,v in train_metrics_final.items()},
        **{f"Val_{k}":   v for k,v in val_metrics_final.items()},
        **{f"Test_{k}":  v for k,v in test_metrics_final.items()},
    }]).to_csv(os.path.join(DS_CKPT_DIR, "final_model_metrics.csv"), index=False)

    ALL_RESULTS[ds_name] = {
        "kfold_df"      : df_results,
        "kfold_avg"     : avg,
        "kfold_std"     : std,
        "all_fold_data" : all_fold_data,
        "final_train"   : train_metrics_final,
        "final_val"     : val_metrics_final,
        "final_test"    : test_metrics_final,
        # keep handles for downstream cells
        "train_preds_proba_final"  : train_preds_proba_final,
        "train_preds_final"        : train_preds_final,
        "val_preds_proba_final"    : val_preds_proba_final,
        "val_preds_final"          : val_preds_final,
        "y_pred_proba_test"        : y_pred_proba_test,
        "y_pred_test_encoded"      : y_pred_test_encoded,
        "y_true_test_encoded"      : y_true_test_encoded,
        "test_image_info_final"    : test_image_info_final,
        "history_dict"             : history_dict,
        "label_encoder"            : label_encoder,
        "X_train_final"            : X_train_final,
        "y_train_final"            : y_train_final,
        "y_val_final"              : y_val_final,
        "final_l2"                 : final_l2,
        "final_pca"                : final_pca,
        "final_model"              : final_model,
        "DS_CKPT_DIR"              : DS_CKPT_DIR,
        "X_raw"                    : X_raw,
    }
    print(f"[{ds_name}] ✅ Done.")

# ── BUG FIX #5: removed ++ ────────────────────────────────────────────────────
print("\n" + "█"*80)
print("CROSS-DATASET SUMMARY — FINAL MODEL TEST METRICS")
print("█"*80)
print(f"\n  {'Dataset':<22} {'Accuracy':>10} {'F1':>8} {'AUC':>8} {'Kappa':>8} {'MCC':>8}")
print(f"  {'─'*66}")
for ds_name, res in ALL_RESULTS.items():    # BUG FIX #5: no ++ here
    fm = res["final_test"]
    print(f"  {ds_name:<22} {fm['Accuracy']:>10.4f} {fm['F1']:>8.4f} "
          f"{fm['AUC']:>8.4f} {fm['Cohen_Kappa']:>8.4f} {fm['MCC']:>8.4f}")


---
## Cell 5 — Visualisations: Training Curves, Metrics Bar Chart, Confusion Matrices, ROC, Sample Predictions

Runs for every dataset stored in `ALL_RESULTS`. Can be re-run safely without retraining.


In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import classification_report, precision_recall_fscore_support, roc_curve, auc
from itertools import cycle

METRICS_LIST = ['Accuracy','Precision','Recall','F1','Cohen_Kappa','MCC','AUC']

for ds_name, res in ALL_RESULTS.items():
    print("\n" + "="*80)
    print(f"VISUALISATIONS — {ds_name.upper()}")
    print("="*80)

    # Unpack
    train_preds_proba_final = res["train_preds_proba_final"]
    train_preds_final       = res["train_preds_final"]
    val_preds_proba_final   = res["val_preds_proba_final"]
    val_preds_final         = res["val_preds_final"]
    y_pred_proba_test       = res["y_pred_proba_test"]
    y_pred_test_encoded     = res["y_pred_test_encoded"]
    y_true_test_encoded     = res["y_true_test_encoded"]
    test_image_info_final   = res["test_image_info_final"]
    history_dict            = res["history_dict"]
    label_encoder           = res["label_encoder"]
    y_train_final           = res["y_train_final"]
    y_val_final             = res["y_val_final"]
    train_metrics_final     = res["final_train"]
    val_metrics_final       = res["final_val"]
    test_metrics_final      = res["final_test"]
    n_classes               = len(label_encoder.classes_)

    # ── Training curves ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(1,2,figsize=(16,5))
    axes[0].plot(history_dict['accuracy'],     lw=2.5, color='#2E86AB', label='Train')
    axes[0].plot(history_dict['val_accuracy'], lw=2.5, color='#A23B72', label='Val')
    axes[0].set_title('Accuracy',fontsize=13,fontweight='bold'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(True,alpha=0.3)
    axes[1].plot(history_dict['loss'],     lw=2.5, color='#2E86AB', label='Train')
    axes[1].plot(history_dict['val_loss'], lw=2.5, color='#A23B72', label='Val')
    axes[1].set_title('Loss',fontsize=13,fontweight='bold'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True,alpha=0.3)
    plt.suptitle(f'Training History — {ds_name}',fontsize=15,fontweight='bold',y=1.02)
    plt.tight_layout(); plt.show()

    # ── Metrics bar chart ────────────────────────────────────────────────────
    x = np.arange(len(METRICS_LIST)); w = 0.25
    fig, ax = plt.subplots(figsize=(16,8))
    b1 = ax.bar(x-w, [train_metrics_final[m] for m in METRICS_LIST], w, label='Training',   alpha=0.85, color='#2ecc71')
    b2 = ax.bar(x,   [val_metrics_final[m]   for m in METRICS_LIST], w, label='Validation', alpha=0.85, color='#3498db')
    b3 = ax.bar(x+w, [test_metrics_final[m]  for m in METRICS_LIST], w, label='Test ', alpha=0.85, color='#3498db')
    ax.set_xticks(x); ax.set_xticklabels(METRICS_LIST,fontsize=11,fontweight='bold')
    ax.set_title(f'Performance Metrics — {ds_name}',fontsize=15,fontweight='bold')
    ax.legend(fontsize=12); ax.grid(True,alpha=0.3,axis='y'); ax.set_ylim([0,1.15])
    for bars in [b1,b2,b3]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2., h+0.01, f'{h:.3f}',
                    ha='center',va='bottom',fontsize=8,fontweight='bold')
    plt.tight_layout(); plt.show()

    # ── Confusion matrices ───────────────────────────────────────────────────
    fig, axes = plt.subplots(1,3,figsize=(24,7))
    for ax, yt, yp, title, cmap in [
        (axes[0], y_train_final,       train_preds_final,    f'Training\nAcc:{train_metrics_final["Accuracy"]:.4f}',    'Greens'),
        (axes[1], y_val_final,         val_preds_final,      f'Validation\nAcc:{val_metrics_final["Accuracy"]:.4f}',    'Blues'),
        (axes[2], y_true_test_encoded, y_pred_test_encoded,  f'Test \nAcc:{test_metrics_final["Accuracy"]:.4f}',   'Blues'),
    ]:
        sns.heatmap(confusion_matrix(yt,yp), annot=True, fmt='d', cmap=cmap,
                    xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_,
                    ax=ax, cbar_kws={'label':'Count'})
        ax.set_title(title,fontsize=12,fontweight='bold')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.suptitle(f'Confusion Matrices — {ds_name}',fontsize=15,fontweight='bold')
    plt.tight_layout(); plt.show()

    # Classification reports
    for split, yt, yp in [
        ("TRAINING",    y_train_final,       train_preds_final),
        ("VALIDATION",  y_val_final,         val_preds_final),
        ("TEST",  y_true_test_encoded, y_pred_test_encoded),
    ]:
        print(f"\n{'='*60}\n{split} — CLASSIFICATION REPORT ({ds_name})\n{'='*60}")
        print(classification_report(yt, yp, target_names=label_encoder.classes_, digits=4))

    # ── Per-class bar charts ──────────────────────────────────────────────────
    tr_p,tr_r,tr_f,_ = precision_recall_fscore_support(y_train_final,       train_preds_final,    average=None, zero_division=0)
    va_p,va_r,va_f,_ = precision_recall_fscore_support(y_val_final,         val_preds_final,      average=None, zero_division=0)
    te_p,te_r,te_f,_ = precision_recall_fscore_support(y_true_test_encoded, y_pred_test_encoded,  average=None, zero_division=0)
    x = np.arange(n_classes); w = 0.25
    fig, axes = plt.subplots(1,3,figsize=(24,7))
    for ax, name, trv, vav, tev in [
        (axes[0],'Precision',tr_p,va_p,te_p),
        (axes[1],'Recall',   tr_r,va_r,te_r),
        (axes[2],'F1-Score', tr_f,va_f,te_f)]:
        ax.bar(x-w, trv, w, label='Train', alpha=0.85, color='#2ecc71')
        ax.bar(x,   vav, w, label='Val',   alpha=0.85, color='#3498db')
        ax.bar(x+w, tev, w, label='Test',  alpha=0.85, color='#e74c3c')
        ax.set_xticks(x); ax.set_xticklabels(label_encoder.classes_, rotation=45, ha='right')
        ax.set_title(f'{name} per Class',fontsize=13,fontweight='bold')
        ax.legend(); ax.grid(True,alpha=0.3,axis='y'); ax.set_ylim([0,1.1])
    plt.suptitle(f'Per-Class Metrics — {ds_name}',fontsize=15,fontweight='bold')
    plt.tight_layout(); plt.show()

    # ── ROC curves ───────────────────────────────────────────────────────────
    y_tr_bin = label_binarize(y_train_final,       classes=range(n_classes))
    y_va_bin = label_binarize(y_val_final,         classes=range(n_classes))
    y_te_bin = label_binarize(y_true_test_encoded, classes=range(n_classes))

    def _roc(y_bin, proba):
        fpr,tpr,auc_d = {},{},{}
        for i in range(n_classes):
            fpr[i],tpr[i],_ = roc_curve(y_bin[:,i],proba[:,i]); auc_d[i]=auc(fpr[i],tpr[i])
        fpr['micro'],tpr['micro'],_ = roc_curve(y_bin.ravel(),proba.ravel())
        auc_d['micro'] = auc(fpr['micro'],tpr['micro']); return fpr,tpr,auc_d

    fpr_tr,tpr_tr,auc_tr = _roc(y_tr_bin, train_preds_proba_final)
    fpr_va,tpr_va,auc_va = _roc(y_va_bin, val_preds_proba_final)
    fpr_te,tpr_te,auc_te = _roc(y_te_bin, y_pred_proba_test)

    fig, axes = plt.subplots(1,3,figsize=(24,7))
    clrs = cycle(['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f'])
    for ax, df, dt, da, title in [
        (axes[0],fpr_tr,tpr_tr,auc_tr,'Training'),
        (axes[1],fpr_va,tpr_va,auc_va,'Validation'),
        (axes[2],fpr_te,tpr_te,auc_te,'Test (TTA)')]:
        for i,color in zip(range(n_classes),clrs):
            ax.plot(df[i],dt[i],color=color,lw=2,label=f'{label_encoder.classes_[i]} (AUC={da[i]:.3f})')
        ax.plot(df['micro'],dt['micro'],'navy',lw=3,ls='--',label=f'Micro (AUC={da["micro"]:.3f})')
        ax.plot([0,1],[0,1],'k--',lw=2,label='Random')
        ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
        ax.set_xlabel('FPR',fontsize=11,fontweight='bold')
        ax.set_ylabel('TPR',fontsize=11,fontweight='bold')
        ax.set_title(f'ROC — {title}',fontsize=13,fontweight='bold')
        ax.legend(loc='lower right',fontsize=8); ax.grid(True,alpha=0.3)
    plt.suptitle(f'Multi-class ROC — {ds_name}',fontsize=15,fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\nPer-Class AUC — Test Set ({ds_name})")
    print(f"  {'Class':<18} {'AUC':>8}")
    for i,cls in enumerate(label_encoder.classes_): print(f"  {cls:<18} {auc_te[i]:>8.4f}")
    print(f"  {'Micro-avg':<18} {auc_te['micro']:>8.4f}")

    # ── Sample predictions ───────────────────────────────────────────────────
    sample_idx = random.sample(range(len(test_image_info_final)), min(15, len(test_image_info_final)))
    n_cols=5; n_rows=(len(sample_idx)+n_cols-1)//n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows)); axes = axes.ravel()
    for idx, si in enumerate(sample_idx):
        s = test_image_info_final[si]
        img = cv2.imread(s['path'])
        if img is not None:
            axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ok = s['true_label']==s['pred_label']
        axes[idx].set_title(
            f"{'✓' if ok else '✗'} True: {s['true_label']}\nPred: {s['pred_label']}\nConf: {s['confidence']:.3f}",
            fontsize=9,fontweight='bold', color='green' if ok else 'red')
        axes[idx].axis('off')
    for idx in range(len(sample_idx),len(axes)): axes[idx].axis('off')
    plt.suptitle(f'Sample Predictions — {ds_name}',fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.show()

    correct = sum(1 for s in test_image_info_final if s['true_label']==s['pred_label'])
    print(f"  Test: {correct}/{len(test_image_info_final)} correct ({correct/len(test_image_info_final):.4f})")


---
## Cell 6 — SHAP Global Explanations

> **Bug fix #6:** SHAP checkpoint path is now per-dataset (`DS_CKPT_DIR/shap_values.npy`)
> instead of a single global path that caused all datasets to overwrite each other.


In [ ]:
import shap

for ds_name, res in ALL_RESULTS.items():
    print("\n" + "="*80)
    print(f"SHAP — {ds_name.upper()}")
    print("="*80)

    DS_CKPT_DIR   = res["DS_CKPT_DIR"]
    # BUG FIX #6: per-dataset paths (not a single global path)
    SHAP_FILE     = os.path.join(DS_CKPT_DIR, "shap_values.npy")
    SHAP_SAMP_FILE= os.path.join(DS_CKPT_DIR, "shap_sample.npy")

    X_train_final = res["X_train_final"]
    final_model   = res["final_model"]
    label_encoder = res["label_encoder"]
    n_classes     = len(label_encoder.classes_)

    shap_sample_size = min(300, len(X_train_final))
    idx_shap         = np.random.choice(len(X_train_final), shap_sample_size, replace=False)
    X_shap_sample    = X_train_final[idx_shap]
    feature_names    = [f"PC_{i+1}" for i in range(X_train_final.shape[1])]

    if os.path.exists(SHAP_FILE):
        print("📂 Loading SHAP values from checkpoint...")
        shap_values   = np.load(SHAP_FILE,      allow_pickle=True)
        X_shap_sample = np.load(SHAP_SAMP_FILE, allow_pickle=True)
        print("✅ Loaded.")
    else:
        try:
            print("→ Trying DeepExplainer...")
            explainer   = shap.DeepExplainer(final_model, X_shap_sample[:50])
            shap_values = explainer.shap_values(X_shap_sample)
            print("✅ DeepExplainer OK.")
        except Exception as e:
            print(f"⚠️  DeepExplainer failed: {e}")
            print("→ KernelExplainer fallback...")
            explainer     = shap.KernelExplainer(final_model.predict, X_shap_sample[:30])
            shap_values   = explainer.shap_values(X_shap_sample[:100])
            X_shap_sample = X_shap_sample[:100]
            print("✅ KernelExplainer OK.")
        np.save(SHAP_FILE,      shap_values)
        np.save(SHAP_SAMP_FILE, X_shap_sample)
        print("💾 SHAP values saved.")

    sv = shap_values[0] if isinstance(shap_values, list) else shap_values

    plt.figure(figsize=(13,6))
    shap.summary_plot(sv, X_shap_sample, feature_names=feature_names, max_display=20, show=False)
    plt.title(f"SHAP Summary (dot) — {ds_name}\n"
              "Each dot = one sample. Colour = feature value. X-axis = SHAP contribution.\n\n\n",
              fontsize=12,fontweight='bold')
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(11,5))
    shap.summary_plot(sv, X_shap_sample, feature_names=feature_names,
                      plot_type="bar", max_display=20, show=False)
    plt.title(f"SHAP Global Feature Importance (bar) — {ds_name}\n"
              "Mean |SHAP value|: longer bar = more influential PCA component.\n\n\n",
              fontsize=12,fontweight='bold')
    plt.tight_layout(); plt.show()

    if isinstance(shap_values, list) and len(shap_values) == n_classes:
        for i, cls in enumerate(label_encoder.classes_):
            plt.figure(figsize=(12,5))
            shap.summary_plot(shap_values[i], X_shap_sample, feature_names=feature_names,
                              plot_type="bar", max_display=15, show=False)
            plt.title(f"SHAP — Class '{cls}' | {ds_name}",fontsize=12,fontweight='bold')
            plt.tight_layout(); plt.show()

    print(f"✅ SHAP complete for {ds_name}.")


---
## Cell 7 — LIME Local Explanations (2 Correct + 2 Incorrect per dataset)


In [ ]:
for ds_name, res in ALL_RESULTS.items():
    print("\n" + "="*80)
    print(f"LIME — {ds_name.upper()}")
    print("="*80)

    final_model           = res["final_model"]
    final_l2              = res["final_l2"]
    final_pca             = res["final_pca"]
    label_encoder         = res["label_encoder"]
    test_image_info_final = res["test_image_info_final"]

    correct   = [s for s in test_image_info_final if s["true_label"]==s["pred_label"]]
    incorrect = [s for s in test_image_info_final if s["true_label"]!=s["pred_label"]]
    lime_targets = (
        [("CORRECT",   s) for s in random.sample(correct,   min(2, len(correct)))] +
        [("INCORRECT", s) for s in random.sample(incorrect, min(2, len(incorrect)))]
    )

    if not lime_targets:
        print("⚠️  No samples available — skipping."); continue

    fig, axes = plt.subplots(len(lime_targets), 3, figsize=(18, 6*len(lime_targets)))
    if len(lime_targets)==1: axes=[axes]

    for row_idx, (status, sample) in enumerate(lime_targets):
        print(f"  Generating LIME for: {sample['name']} [{status}]...")
        explanation, img_rgb, _ = generate_lime_explanation(
            final_model, sample["path"], final_l2, final_pca,
            label_encoder, num_samples=CONFIG["lime_samples"])

        if explanation is None:
            print(f"  ⚠️  LIME failed — skipping."); continue

        top_label = label_encoder.transform([sample["pred_label"]])[0]
        c = "green" if status=="CORRECT" else "red"

        axes[row_idx][0].imshow(img_rgb)
        axes[row_idx][0].set_title(
            f"[{status}]\nTrue: {sample['true_label']} | Pred: {sample['pred_label']}\n"
            f"Confidence: {sample['confidence']:.3f}",
            fontsize=11,fontweight="bold",color=c)
        axes[row_idx][0].axis("off")

        temp1, mask1 = explanation.get_image_and_mask(
            top_label, positive_only=True, num_features=10, hide_rest=False)
        axes[row_idx][1].imshow(mark_boundaries(temp1, mask1))
        axes[row_idx][1].set_title("Supporting regions (green)",fontsize=11)
        axes[row_idx][1].axis("off")

        temp2, mask2 = explanation.get_image_and_mask(
            top_label, positive_only=False, num_features=10, hide_rest=False)
        axes[row_idx][2].imshow(mark_boundaries(temp2, mask2))
        axes[row_idx][2].set_title("All regions: green=support, red=contradict",fontsize=11)
        axes[row_idx][2].axis("off")

    plt.suptitle(f"LIME Explanations — {ds_name}",fontsize=14,fontweight="bold")
    plt.tight_layout(); plt.show()
    print(f"✅ LIME complete for {ds_name}.")


---
## Cell 8 — PCA Dimensionality Report & Inference Metrics


In [ ]:
import time, tracemalloc, platform, psutil

for ds_name, res in ALL_RESULTS.items():
    print("\n" + "="*70)
    print(f"PCA REPORT + INFERENCE — {ds_name.upper()}")
    print("="*70)

    all_fold_data = res["all_fold_data"]
    final_pca     = res["final_pca"]
    final_l2      = res["final_l2"]
    final_model   = res["final_model"]
    X_raw         = res["X_raw"]
    test_info     = res["test_image_info_final"]

    fold_ks = [fd["fold_pca"].n_components_ for fd in all_fold_data]
    print(f"  K-Fold PCA k per fold   : {fold_ks}")
    print(f"  K-Fold PCA k mean/range : {sum(fold_ks)/len(fold_ks):.1f} / [{min(fold_ks)}, {max(fold_ks)}]")
    print(f"  Final model PCA k       : {final_pca.n_components_}")
    print(f"  Raw CNN feature dim     : {X_raw.shape[1]}")
    print(f"  Reduction factor        : {X_raw.shape[1]/final_pca.n_components_:.1f}×")

    # Build timing images
    imgs_timing = []
    for s in test_info[:50]:
        img = cv2.imread(s["path"])
        if img is not None:
            imgs_timing.append(cv2.resize(img, CONFIG["input_shape"][:2]))

    if imgs_timing:
        for _ in range(3):   # warmup
            _f = extract_single_image_features(imgs_timing[0])
            final_model.predict(final_pca.transform(final_l2.transform([_f])), verbose=0)

        tracemalloc.start(); snap0 = tracemalloc.take_snapshot()
        lat_ms = []; t0_tot = time.perf_counter()
        for img in imgs_timing:
            t0 = time.perf_counter()
            _f = extract_single_image_features(img)
            final_model.predict(final_pca.transform(final_l2.transform([_f])), verbose=0)
            lat_ms.append((time.perf_counter()-t0)*1000)
        tot_s = time.perf_counter()-t0_tot
        snap1 = tracemalloc.take_snapshot(); tracemalloc.stop()

        mem_mb = sum(s.size_diff for s in snap1.compare_to(snap0,'lineno')) / 1e6
        rss_mb = psutil.Process(os.getpid()).memory_info().rss / 1e6
        lats   = sorted(lat_ms)
        mean_l = sum(lat_ms)/len(lat_ms)

        try:
            import subprocess
            gpu = subprocess.check_output(
                ["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                stderr=subprocess.DEVNULL).decode().strip()
        except Exception: gpu = "N/A"

        print(f"\n  CPU: {platform.processor() or 'N/A'}  RAM: {psutil.virtual_memory().total/1e9:.1f} GB  GPU: {gpu}")
        print(f"  Latency — Mean:{mean_l:.1f}ms  P50:{lats[len(lats)//2]:.1f}ms  "
              f"P95:{lats[int(len(lats)*.95)]:.1f}ms  Min:{min(lat_ms):.1f}ms  Max:{max(lat_ms):.1f}ms")
        print(f"  Throughput : {1000/mean_l:.1f} FPS   Total({len(imgs_timing)} imgs): {tot_s:.2f}s")
        print(f"  Memory delta: {mem_mb:+.2f} MB   Process RSS: {rss_mb:.1f} MB")

        # ── Store for paper reporting ──────────────────────────────────────
        res["inference_metrics"] = {
            "n_images"      : len(imgs_timing),
            "mean_lat_ms"   : round(mean_l, 2),
            "p50_lat_ms"    : round(lats[len(lats)//2], 2),
            "p95_lat_ms"    : round(lats[int(len(lats)*.95)], 2),
            "min_lat_ms"    : round(min(lat_ms), 2),
            "max_lat_ms"    : round(max(lat_ms), 2),
            "fps"           : round(1000 / mean_l, 1),
            "total_time_s"  : round(tot_s, 2),
            "mem_delta_mb"  : round(mem_mb, 2),
            "rss_mb"        : round(rss_mb, 1),
            "hw_cpu"        : platform.processor() or platform.machine(),
            "hw_ram_gb"     : round(psutil.virtual_memory().total / 1e9, 1),
            "hw_gpu"        : gpu,
            "hw_os"         : platform.platform(),
            "hw_python"     : platform.python_version(),
            "hw_tf"         : tf.__version__,
        }


---
## Cell 9 — Save Model & Per-Dataset Results to Drive


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

for ds_name, res in ALL_RESULTS.items():
    SAVE_DIR = os.path.join(res["DS_CKPT_DIR"], "results")
    os.makedirs(SAVE_DIR, exist_ok=True)
    label_encoder = res["label_encoder"]

    try:
        res["final_model"].save(os.path.join(SAVE_DIR, "final_model.h5"))
        print(f"[{ds_name}] ✅ Model saved.")

        pd.DataFrame([{
            "Dataset": ds_name,
            **{f"Train_{k}": v for k,v in res["final_train"].items()},
            **{f"Val_{k}":   v for k,v in res["final_val"].items()},
            **{f"Test_{k}":  v for k,v in res["final_test"].items()},
        }]).to_csv(os.path.join(SAVE_DIR, "final_model_metrics.csv"), index=False)

        pd.DataFrame(res["test_image_info_final"]).to_csv(
            os.path.join(SAVE_DIR, "final_test_predictions.csv"), index=False)

        te_p,te_r,te_f,_ = precision_recall_fscore_support(
            res["y_true_test_encoded"], res["y_pred_test_encoded"],
            average=None, zero_division=0)
        pd.DataFrame({
            "Class": label_encoder.classes_,
            "Test_Precision": te_p, "Test_Recall": te_r, "Test_F1": te_f,
        }).to_csv(os.path.join(SAVE_DIR, "final_per_class_metrics.csv"), index=False)

        print(f"[{ds_name}] ✅ All results saved to {SAVE_DIR}")
    except Exception as e:
        print(f"[{ds_name}] ⚠️  Save error: {e}")


---
## Cell 10 — Combined XLSX Summary (K-Fold + Final Model across all datasets)

Writes one `.xlsx` file to Drive with:
- **Sheet: `Summary`** — one row per dataset, all final test metrics
- **Sheet: `KFold_<dataset>`** — per-fold metrics for each dataset  
- **Sheet: `Final_<dataset>`** — train/val/test metrics for each dataset


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

XLSX_PATH = os.path.join(CKPT_ROOT, "ALL_DATASETS_SUMMARY.xlsx")

wb = Workbook()

# ── Style helpers ─────────────────────────────────────────────────────────────
HDR_FILL  = PatternFill("solid", start_color="1F4E79", end_color="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
ALT_FILL  = PatternFill("solid", start_color="D6E4F0", end_color="D6E4F0")
WH_FILL   = PatternFill("solid", start_color="FFFFFF", end_color="FFFFFF")
VAL_FONT  = Font(name="Arial", size=10)
BOLD_FONT = Font(name="Arial", bold=True, size=10)
CENTER    = Alignment(horizontal="center", vertical="center")
THIN      = Side(style="thin", color="CCCCCC")
BORDER    = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def style_header_row(ws, row, ncols):
    for c in range(1, ncols+1):
        cell = ws.cell(row=row, column=c)
        cell.fill = HDR_FILL; cell.font = HDR_FONT
        cell.alignment = CENTER; cell.border = BORDER

def style_data_row(ws, row, ncols, alt=False):
    fill = ALT_FILL if alt else WH_FILL
    for c in range(1, ncols+1):
        cell = ws.cell(row=row, column=c)
        cell.fill = fill; cell.font = VAL_FONT
        cell.alignment = CENTER; cell.border = BORDER

def auto_col_width(ws):
    for col in ws.columns:
        max_len = max((len(str(cell.value)) if cell.value else 0 for cell in col), default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 30)

METRICS_LIST = ["Accuracy","Precision","Recall","F1","Cohen_Kappa","MCC","AUC"]

# ══════════════════════════════════════════════════════════════
# Sheet 1: Summary — one row per dataset
# ══════════════════════════════════════════════════════════════
ws_sum = wb.active; ws_sum.title = "Summary"
ws_sum.freeze_panes = "B2"

# Header
headers = ["Dataset"] + [f"KFold_Test_{m}" for m in METRICS_LIST] +           [f"Final_Train_{m}" for m in METRICS_LIST] +           [f"Final_Val_{m}"   for m in METRICS_LIST] +           [f"Final_Test_{m}"  for m in METRICS_LIST]
for c, h in enumerate(headers, 1):
    ws_sum.cell(row=1, column=c).value = h
style_header_row(ws_sum, 1, len(headers))

for r_idx, (ds_name, res) in enumerate(ALL_RESULTS.items(), 2):
    avg = res["kfold_avg"]
    row = [ds_name] +           [round(avg[f"Test_{m}"], 4) for m in METRICS_LIST] +           [round(res["final_train"][m], 4) for m in METRICS_LIST] +           [round(res["final_val"][m],   4) for m in METRICS_LIST] +           [round(res["final_test"][m],  4) for m in METRICS_LIST]
    for c, v in enumerate(row, 1):
        ws_sum.cell(row=r_idx, column=c).value = v
    style_data_row(ws_sum, r_idx, len(headers), alt=(r_idx % 2 == 0))

auto_col_width(ws_sum)
ws_sum.row_dimensions[1].height = 28

# ══════════════════════════════════════════════════════════════
# Sheet per dataset: K-Fold results
# ══════════════════════════════════════════════════════════════
for ds_name, res in ALL_RESULTS.items():
    safe = ds_name.replace("-","_")[:20]
    ws = wb.create_sheet(title=f"KFold_{safe}")
    ws.freeze_panes = "B2"

    kfold_df = res["kfold_df"]
    cols = list(kfold_df.columns)
    for c, h in enumerate(cols, 1):
        ws.cell(row=1, column=c).value = h
    style_header_row(ws, 1, len(cols))

    for r_idx, (_, row_data) in enumerate(kfold_df.iterrows(), 2):
        for c, v in enumerate(row_data, 1):
            cell = ws.cell(row=r_idx, column=c)
            cell.value = round(v, 4) if isinstance(v, float) else v
        style_data_row(ws, r_idx, len(cols), alt=(r_idx % 2 == 0))

    # Mean ± Std rows
    avg = res["kfold_avg"]; std = res["kfold_std"]
    n_rows = len(kfold_df) + 2
    ws.cell(row=n_rows,   column=1).value = "Mean"
    ws.cell(row=n_rows+1, column=1).value = "Std"
    ws.cell(row=n_rows,   column=1).font  = BOLD_FONT
    ws.cell(row=n_rows+1, column=1).font  = BOLD_FONT
    for c, col in enumerate(cols[1:], 2):
        ws.cell(row=n_rows,   column=c).value = round(avg[col], 4) if col in avg else ""
        ws.cell(row=n_rows+1, column=c).value = round(std[col], 4) if col in std else ""
        for rr in [n_rows, n_rows+1]:
            ws.cell(row=rr, column=c).font      = VAL_FONT
            ws.cell(row=rr, column=c).alignment = CENTER
            ws.cell(row=rr, column=c).border    = BORDER

    auto_col_width(ws)
    ws.row_dimensions[1].height = 28

# ══════════════════════════════════════════════════════════════
# Sheet per dataset: Final model metrics
# ══════════════════════════════════════════════════════════════
for ds_name, res in ALL_RESULTS.items():
    safe = ds_name.replace("-","_")[:20]
    ws = wb.create_sheet(title=f"Final_{safe}")
    ws.freeze_panes = "B2"

    headers_f = ["Split"] + METRICS_LIST
    for c, h in enumerate(headers_f, 1):
        ws.cell(row=1, column=c).value = h
    style_header_row(ws, 1, len(headers_f))

    for r_idx, (split, mdict) in enumerate([
        ("Train",     res["final_train"]),
        ("Val",       res["final_val"]),
        ("Test (TTA)",res["final_test"])], 2):
        ws.cell(row=r_idx, column=1).value = split
        for c, m in enumerate(METRICS_LIST, 2):
            ws.cell(row=r_idx, column=c).value = round(mdict[m], 4)
        style_data_row(ws, r_idx, len(headers_f), alt=(r_idx % 2 == 0))

    auto_col_width(ws)
    ws.row_dimensions[1].height = 28

wb.save(XLSX_PATH)
print(f"\n✅ Combined metrics workbook saved → {XLSX_PATH}")
print(f"   Sheets: Summary + {len(ALL_RESULTS)} KFold sheets + {len(ALL_RESULTS)} Final sheets")


---
## Cell 11 — Final Cross-Dataset Comparison

Quick printed summary of all datasets side by side.


In [ ]:
print("\n" + "█"*80)
print("CROSS-DATASET COMPARISON — K-FOLD & FINAL MODEL TEST METRICS")
print("█"*80)

print(f"\n  K-FOLD (mean test):")
print(f"  {'Dataset':<24} {'Accuracy':>10} {'F1':>8} {'AUC':>8} {'Kappa':>8} {'MCC':>8}")
print(f"  {'─'*68}")
for ds_name, res in ALL_RESULTS.items():
    avg = res["kfold_avg"]
    print(f"  {ds_name:<24} {avg['Test_Accuracy']:>10.4f} {avg['Test_F1']:>8.4f} "
          f"{avg['Test_AUC']:>8.4f} {avg['Test_Cohen_Kappa']:>8.4f} {avg['Test_MCC']:>8.4f}")

print(f"\n  FINAL MODEL (test with TTA):")
print(f"  {'Dataset':<24} {'Accuracy':>10} {'F1':>8} {'AUC':>8} {'Kappa':>8} {'MCC':>8}")
print(f"  {'─'*68}")
for ds_name, res in ALL_RESULTS.items():
    fm = res["final_test"]
    print(f"  {ds_name:<24} {fm['Accuracy']:>10.4f} {fm['F1']:>8.4f} "
          f"{fm['AUC']:>8.4f} {fm['Cohen_Kappa']:>8.4f} {fm['MCC']:>8.4f}")

print(f"\n✅ All results saved to: {CKPT_ROOT}")
print(f"   Combined xlsx: {CKPT_ROOT}/ALL_DATASETS_SUMMARY.xlsx")


In [ ]:
cat /proc/cpuinfo | grep "model name"